12/6/2025 - Moosa

Purpose: The goal is to understand how each model performs on unseen data and save all evaluation outputs to the correct project folders for team reference and presentation.

In [4]:
import sys; 
import pandas as pd
import pickle
import os
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
sys.path.append("../../") #To allow imports from parent folders

# Load
df = pd.read_csv("../data/cleaned/wednesday_cleaned.csv")
X = df.drop(['Label', 'Attack'], axis=1)
y = df['Attack']

# Train/test split 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale test set
scaler = StandardScaler()
scaler.fit(X_train)
X_test_scaled = scaler.transform(X_test)

# Models loading
models = {
    "random_forest": "../models/saved_model/random_forest.pkl",
    "decision_tree": "../models/saved_model/decision_tree.pkl",
    "svm": "../models/saved_model/svm.pkl"
}

loaded = {}
for name, path in models.items():
    with open(path, "rb") as f:
        loaded[name] = pickle.load(f)

# Output folders
os.makedirs("../models/evaluation/confusion_matrices", exist_ok=True)
os.makedirs("../models/evaluation/metrics_tables", exist_ok=True)

# Evaluate and save outputs
for name, model in loaded.items():
    print(f"\n{name.upper()} \n")

    y_pred = model.predict(X_test_scaled)

    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    print("\nConfusion Matrix:\n", cm)

    with open(f"../models/evaluation/confusion_matrices/{name}_cm.txt", "w") as f:
        f.write(str(cm))

    # Classification report
    report = classification_report(y_test, y_pred)
    print("\nClassification Report:\n", report)

    with open(f"../models/evaluation/metrics_tables/{name}_report.txt", "w") as f:
        f.write(report)

print("\nAll evaluations completed and saved!!")


/opt/anaconda3/envs/ml/lib/python3.11/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.7.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/opt/anaconda3/envs/ml/lib/python3.11/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.7.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/opt/anaconda3/envs/ml/lib/python3.11/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator SVC from version 1.7.1 when using version 1.


RANDOM_FOREST 


Confusion Matrix:
 [[  998     1]
 [    1 11201]]

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00       999
           1       1.00      1.00      1.00     11202

    accuracy                           1.00     12201
   macro avg       1.00      1.00      1.00     12201
weighted avg       1.00      1.00      1.00     12201


DECISION_TREE 


Confusion Matrix:
 [[  996     3]
 [    2 11200]]

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00       999
           1       1.00      1.00      1.00     11202

    accuracy                           1.00     12201
   macro avg       1.00      1.00      1.00     12201
weighted avg       1.00      1.00      1.00     12201


SVM 


Confusion Matrix:
 [[  943    56]
 [    5 11197]]

Classification Report:
               precision    recall  f1-score   support

           0       0.9

Evaluated three complex models (Random Forest, Decision Tree, and SVM) using the test set and generated both confusion matrices and classification reports. All models performed extremely well overall, with Random Forest achieving perfect accuracy and the other two models showing only a few misclassifications. 

The confusion matrices were saved in models/evaluation/confusion_matrices/ and the classification reports were saved in models/evaluation/metrics_tables/ for the team to use in later steps.

## Task Complete

In [7]:
import pickle
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

# Load all saved models
models = {
    'Naive Bayes': pickle.load(open('../models/saved_model/naive_bayes.pkl', 'rb')),
    'kNN': pickle.load(open('../models/saved_model/knn.pkl', 'rb')),
    'Logistic Regression': pickle.load(open('../models/saved_model/logistic_regression.pkl', 'rb')),
    'Random Forest': pickle.load(open('../models/saved_model/random_forest.pkl', 'rb')),
    'Decision Tree': pickle.load(open('../models/saved_model/decision_tree.pkl', 'rb')),
    'SVM': pickle.load(open('../models/saved_model/svm.pkl', 'rb'))
}

# Generate ROC curve for each model
for model_name, model in models.items():
    try:
        y_proba = model.predict_proba(X_test)[:, 1]  # For most models
    except AttributeError:
        y_proba = model.decision_function(X_test)     # For SVM
    
    # Calculate ROC curve
    fpr, tpr, thresholds = roc_curve(y_test, y_proba)
    roc_auc = auc(fpr, tpr)
    
    # Plot
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='blue', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], color='gray', lw=1, linestyle='--', label='Random Classifier')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'ROC Curve - {model_name}')
    plt.legend(loc="lower right")
    plt.grid(alpha=0.3)
    
    # Save
    filename = model_name.lower().replace(' ', '_')
    plt.savefig(f'../models/evaluation/roc_curves/{filename}_roc.png', dpi=300, bbox_inches='tight')
    plt.close()
    

/opt/anaconda3/envs/ml/lib/python3.11/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.7.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/opt/anaconda3/envs/ml/lib/python3.11/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.7.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/opt/anaconda3/envs/ml/lib/python3.11/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator SVC from version 1.7.1 when using version 1.

In [8]:
import json
import pandas as pd

# Load final_features.json
with open('../data/final_features.json', 'r') as f:
    schema = json.load(f)

# Check 1: Verify feature count
scaler_features = schema['scaler_params']['feature_names']
print(f"Features in schema: {len(scaler_features)}")
print(f"Features in training data: {X_train.shape[1]}")
assert len(scaler_features) == X_train.shape[1], "Feature count mismatch!"

# Check 2: Verify feature order matches
schema_order = schema['scaler_params']['feature_names']
model_order = X_train.columns.tolist()
assert schema_order == model_order, "Feature order mismatch!"
print("Feature order matches")

# Check 3: Verify numeric types
print("\nChecking data types:")
for feature in scaler_features[:5]:  # Show first 5
    dtype = schema[feature]['dtype']
    print(f"  {feature}: {dtype}")


Features in schema: 67
Features in training data: 67
Feature order matches

Checking data types:
  Destination_Port: int64
  Flow_Duration: int64
  Total_Fwd_Packets: int64
  Total_Backward_Packets: int64
  Total_Length_of_Fwd_Packets: int64


12/08/2025 - Nafisa

**ROC Curves Generated:**
- Created ROC curves for all 6 models (Naive Bayes, kNN, Logistic Regression, Random Forest, Decision Tree, SVM)
- Saved to `models/evaluation/roc_curves/`
- Each curve shows True Positive Rate vs False Positive Rate with AUC score

**Schema Validation:**
- Verified `final_features.json` contains correct 67 features
- Confirmed feature order matches training data
- Validated all features have correct numeric types

### Done with Task